In [126]:
import os, random, math, pickle
import pandas as pd
import numpy as np
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from torch.utils.data import random_split
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

# Set environment variables for reproducibility and safety
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import precision_score, recall_score, accuracy_score

# 1. Configuration & Seeding
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [127]:
# name = 'book'
# N_STATIC_RELATION = 20
# N_CLUSTER = 15

# USER_START_ID = 1
# USER_END_ID = 2890
# ITEM_START_ID = 2891  # book: 2891 
# ITEM_END_ID = 11584   
# STATIC_ENTITY_START_ID = 11585  # book: 2891 
# STATIC_ENTITY_END_ID = 38885  

# N_ENTITY = STATIC_ENTITY_END_ID
# N_RELATION = N_STATIC_RELATION + N_CLUSTER

In [128]:
name = 'movie'
N_STATIC_RELATION = 20
N_CLUSTER = 15

USER_START_ID = 1
USER_END_ID = 1439
ITEM_START_ID = 1440  # book: 2891 
ITEM_END_ID = 6335   
STATIC_ENTITY_START_ID = 6336  # book: 2891 
STATIC_ENTITY_END_ID = 54535  

N_ENTITY = STATIC_ENTITY_END_ID
N_RELATION = N_STATIC_RELATION + N_CLUSTER

#### 1.1 KGCN Model

In [129]:
from collections import defaultdict

class KnownledgeGraph():
    def __init__(self, graph_df, nbr_sample_size=1):
        self.graph_df = graph_df
        self.graph_nbr_sample_size = nbr_sample_size
        self.graph_entities = None
        self.graph_relations = None
        # self.graph_n_entity = 0
        # self.graph_n_relation = 0
        self.graph = defaultdict(set)

    def build(self):
        df = self.graph_df

        # head_id, relation_id, tail_id, head_id:token, relation_id:token, tail_id:token
        head_ids = df['head_id'].values
        tail_ids = df['tail_id'].values

        # self.graph_entities = np.unique(np.concatenate([head_ids, tail_ids]))
        # self.graph_n_entity = len(self.graph_entities)

        # self.graph_relations = np.unique(df['relation_id'].values)
        # self.graph_n_relation = len(self.graph_relations)

        # Build the knowledge graph with sets
        for head_id, relation_id, tail_id, _, _, _ in df.values:
            self.graph[head_id].add((tail_id, relation_id))
            self.graph[tail_id].add((head_id, relation_id))

    def get_neighbors(self, entity_id, sample_size=8):
        """Get neighbors for an entity with optional sampling"""
        if sample_size is None:
            sample_size = self.graph_nbr_sample_size

        entity_id = entity_id.item() if torch.is_tensor(entity_id) else entity_id
        neighbors = list(self.graph.get(entity_id, set()))
        n_neighbors = len(neighbors)

        if n_neighbors == 0:
            return torch.tensor([], dtype=torch.long), torch.tensor([], dtype=torch.long)

        if n_neighbors >= sample_size:
            sampled_indices = np.random.choice(n_neighbors, size=sample_size, replace=False)
        else:
            sampled_indices = np.random.choice(n_neighbors, size=sample_size, replace=True)

        sampled_neighbors = [neighbors[i] for i in sampled_indices]
        adj_entities = torch.tensor([n for n, _ in sampled_neighbors], dtype=torch.long)
        adj_relations = torch.tensor([r for _, r in sampled_neighbors], dtype=torch.long)

        return adj_entities, adj_relations

In [130]:
class SumAggregator(nn.Module):
    def __init__(self, embedding_dim):
        super(SumAggregator, self).__init__()
        self.linear = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, neighbor_embs, central_embs):
        """
        neighbor_embs: Tensor of shape (batch, emb_dim)
        central_embs: Tensor of shape (batch, emb_dim)
        """

        # Combine with central entity embedding
        combined = neighbor_embs + central_embs  # shape: (batch, emb_dim) ([1204, 64])

        # Linear + activation
        output = torch.tanh(self.linear(combined))  # shape: (batch, emb_dim) torch.Size([1204, 64])
        return output

In [131]:
class KGCN(pl.LightningModule):
    def __init__(self, graph: KnownledgeGraph, pos_item_pool, num_neg_samples, embedding_dim, lr):
        super().__init__()
        self.save_hyperparameters()

        model_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_device = model_device

        self.graph = graph
        self.pos_item_pool = pos_item_pool
        self.num_neg_samples = num_neg_samples

        # self.num_users = num_users
        # self.num_items = num_items

        self.embedding_dim = embedding_dim

        # Embedding layersnum_neg_samples
        self.entity_embedding = nn.Embedding(N_ENTITY + 1, embedding_dim, padding_idx = 0).to(model_device)
        self.relation_embedding = nn.Embedding(N_RELATION + 1, embedding_dim, padding_idx = 0).to(model_device)

        # self.user_embedding = nn.Embedding(num_users, embedding_dim).to(model_device)
        # self.entity_embedding = nn.Embedding(graph.graph_n_entity, embedding_dim).to(model_device)
        # self.relation_embedding = nn.Embedding(graph.graph_n_relation, embedding_dim).to(model_device)

        # Aggregator
        self.aggregator = SumAggregator(embedding_dim).to(model_device)

        self.dropout = nn.Dropout(p=0.1)

    def setup(self, stage=None):
        self.train_user_pos_items = self.trainer.datamodule.train_user_pos_items
        self.val_user_pos_items = self.trainer.datamodule.val_user_pos_items

    @staticmethod
    def hit_at_k(pred_items, true_items, k):
        hits = 0
        for pred, true in zip(pred_items, true_items):
            # Count if any true item is in top-k predictions
            if len(set(pred[:k]) & set(true)) > 0:
                hits += 1
        return hits / len(true_items)

    @staticmethod
    def ndcg_at_k(pred_items, true_items, k):
        ndcg = 0.0
        for pred, true in zip(pred_items, true_items):
            gains = []
            for idx, item in enumerate(pred[:k]):
                gains.append(1 if item in true else 0)
            ideal_gains = [1] * min(len(true), k)
            dcg = sum(g / math.log2(i+2) for i, g in enumerate(gains))
            idcg = sum(g / math.log2(i+2) for i, g in enumerate(ideal_gains))
            ndcg += dcg / idcg if idcg > 0 else 0
        return ndcg / len(true_items)

    @staticmethod
    def recall_at_k(pred_items, true_items, k):
        recall = 0.0
        for pred, true in zip(pred_items, true_items):
            recall += len(set(pred[:k]) & set(true)) / len(true)
        return recall / len(true_items)

    @staticmethod
    def precision_at_k(pred_items, true_items, k):
        precision = 0.0
        for pred, true in zip(pred_items, true_items):
            precision += len(set(pred[:k]) & set(true)) / k
        return precision / len(true_items)

    def forward(self, batch):
        user_ids, item_ids, neighbor_ids, relation_ids = batch

        # # full_user_embs = self.user_embedding.weight
        # full_entity_embs = self.entity_embedding.weight
        # full_relation_embs = self.relation_embedding.weight

        user_embs = self.entity_embedding(user_ids) #torch.Size([1204, 64])
        neighbor_embs = self.entity_embedding(neighbor_ids) # torch.Size([1204, 8, 64])
        relation_embs = self.relation_embedding(relation_ids) #torch.Size([1204, 8, 64])

        item_embs = self.entity_embedding(item_ids) #torch.Size([1204, 64])

        ################ User - Relation Attention ################
        # Expand user_embs to match neighbor dimension
        user_embs_expanded = user_embs.unsqueeze(1)  # [1204, 1, 64]

        # Compute scores: dot product between relation_embs and user_embs
        scores = torch.exp(-torch.abs(user_embs_expanded - relation_embs).sum(dim=2))  # [batch_size, k]

        attention_weights = F.softmax(scores, dim=1).unsqueeze(-1)  # [1204, 8, 1]
        weighted_neighbor_embs = neighbor_embs * attention_weights  # [1204, 8, 64]
        weighted_neighbor_embs = weighted_neighbor_embs.mean(dim=1) # [1204, 64]
        ################ User - Relation Attention ################

        updated_item_embs  = self.aggregator(weighted_neighbor_embs, item_embs) #([1204, 64])

        return user_embs, updated_item_embs

    def compute_loss(self, batch, user_embs, updated_item_embs):
        user_ids, item_ids, neighbor_ids, relation_ids = batch
        batch_size = user_ids.size(0)

        pos_item_embs = updated_item_embs

        # dist = torch.abs(user_embs - pos_item_embs).sum(dim=1)
        # pos_scores = torch.exp(-dist)
        pos_scores = (user_embs * pos_item_embs).sum(dim=1)

        # ######################################### Hard negative Sampling #########################################
        # full_item_embs = self.entity_embedding.weight[ITEM_START_ID : ITEM_END_ID + 1]

        # distances = torch.cdist(user_embs, full_item_embs, p=1) # [batch_size, num_items] torch.Size([1204, 4779])
        # scores = torch.exp(-distances)  # [batch_size, num_items] torch.Size([1204, 4779])


        # # ################## Mask all pos_item_ids of the user in training_set ##################
        # ### Basically, the  model should only see the information in the train_dataset.
        # ### Therefore, only mask the pos_item_ids of the user in train_dataset
        # ### All cell (user, item) in val_dataset should be treated as blank hence don't mask the val_dataset
        # for i, u in enumerate(user_ids.tolist()):
        #     pos_item_ids = [item - ITEM_START_ID for item in self.train_user_pos_items[u]]
        #     scores[i, pos_item_ids] = float('-inf')
        # ################## Mask all pos_item_ids of the user in training_set ##################


        ######################################### Hard negative Sampling ########################################
        # full_item_embs = self.entity_embedding.weight[torch.tensor(self.pos_item_pool)]

        # distances = torch.cdist(user_embs, full_item_embs, p=1) # [batch_size, num_items] torch.Size([1204, 4779])
        # scores = torch.exp(-distances)  # [batch_size, num_items] torch.Size([1204, 4779])


        # ################## Mask all pos_item_ids of the user in training_set ##################
        ### Basically, the  model should only see the information in the train_dataset.
        ### Therefore, only mask the pos_item_ids of the user in train_dataset
        ### All cell (user, item) in val_dataset should be treated as blank hence don't mask the val_dataset
        # for i, u in enumerate(user_ids.tolist()):
        #     pos_item_ids = [item - ITEM_START_ID for item in self.train_user_pos_items[u]]
        #     scores[i, pos_item_ids] = float('-inf')
        ################## Mask all pos_item_ids of the user in training_set ##################

        # k = 10 # Select top-K most negatives for each user
        # neg_item_ids = torch.topk(scores, k=k, dim=1).indices 
        # neg_item_ids = [self.pos_item_pool[x] for x in neg_item_ids]# [batch_size, k]

        # 3. Tạo các index ngẫu nhiên từ 0 đến len(item_pool) - 1
        # Kích thước là (num_users, 10)
        random_indices = torch.randint(0, len(self.pos_item_pool), (batch_size, self.num_neg_samples))

        # 4. Dùng các index này để lấy giá trị ID thực tế từ item_pool
        neg_item_ids = self.pos_item_pool[random_indices]


 
        # Get embeddings for these negatives
        neg_item_embs = self.entity_embedding(torch.tensor(neg_item_ids))  # shape: [batch_size, k, emb_dim]

        # neg_scores = torch.exp(-torch.abs(user_embs.unsqueeze(1) - neg_emb).sum(dim=2))  # [batch_size, k]
        # neg_scores = neg_scores.mean(dim=1) # [batch_size]            
        neg_scores = (user_embs.unsqueeze(1) * neg_item_embs).sum(dim=2)
        neg_scores = neg_scores.mean(dim=1)
        ######################################### Hard negative Sampling #########################################

        ####################### Compute Loss #######################
        # scores = torch.cat([pos_scores, neg_scores], dim=0)
        # labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)], dim=0)

        # loss = F.binary_cross_entropy(scores, labels)
        loss = - F.logsigmoid(pos_scores - neg_scores).mean()
        ####################### Compute Loss #######################

        return loss, pos_scores, neg_scores

    def training_step(self, batch, batch_idx):
        user_embs, updated_item_embs = self(batch)
        train_loss, _, _ = self.compute_loss(batch, user_embs, updated_item_embs)

        self.log("train_loss", train_loss, on_epoch=True, prog_bar=True)

        return train_loss

    def validation_step(self, batch, batch_idx):
        user_ids, item_ids, neighbor_ids, relation_ids = batch

        user_embs, updated_item_embs = self(batch)

        val_loss, _, _ = self.compute_loss(batch, user_embs, updated_item_embs)
        
        # full_item_embs = agg_full_entity_embs[:self.num_items]
        full_item_embs = self.entity_embedding.weight[ITEM_START_ID : ITEM_END_ID + 1]

        # distances = torch.cdist(user_embs, full_item_embs, p=1)  # [batch_size, num_items] torch.Size([1204, 4779])
        # scores = torch.exp(-distances)  # [batch_size, num_items] torch.Size([1204, 4779]), it is the score between the ith user in batch_size and ALL items

        scores = (user_embs.unsqueeze(1) * full_item_embs).sum(dim=2)

        # ########## Mask those user-item pair that already in training set so that it won't suggest again
        mask = torch.zeros_like(scores, dtype=torch.bool)
        for i, u in enumerate(user_ids.tolist()):
            trained_items = [item - ITEM_START_ID for item in self.train_user_pos_items[u]]
            # mask[i, trained_items] = True
            scores[i, trained_items] = float('-inf')

        # scores = scores.masked_fill(mask, float('-inf'))    #### Make them to -inf so that TopK won't pick again
        ########## Mask those user-item pair that already in training set so that it won't suggest again


        ################ Calculate metrics
        k_values = [10]  # Example: you can add more values as needed

        for k in k_values:
            # Get top-k items for this k
            topk_items = torch.topk(scores, k=k, dim=1).indices.tolist()
            # topk_items = (topk_indices + ITEM_START_ID).tolist()

            true_items = []  # (1024, variable length), each user may have multiple positive items
            for u in user_ids.tolist():
                adjusted_val_items = [item - ITEM_START_ID for item in self.val_user_pos_items[u]]
                true_items.append(adjusted_val_items)

            # Compute metrics for this k
            hit = self.hit_at_k(topk_items, true_items, k)
            ndcg = self.ndcg_at_k(topk_items, true_items, k)
            recall = self.recall_at_k(topk_items, true_items, k)
            precision = self.precision_at_k(topk_items, true_items, k)

            # Log metrics dynamically
            self.log(f"val_hit@{k:02d}", hit, prog_bar=True)
            self.log(f"val_recall@{k:02d}", recall, prog_bar=True)
            self.log(f"val_precision@{k:02d}", precision, prog_bar=True)
            self.log(f"val_ndcg@{k:02d}", ndcg, prog_bar=True)



        scores = scores.mean(dim=1)
        self.log('val_loss', val_loss, prog_bar=True, logger=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.75)
        return [optimizer], [scheduler]

#### 1.2 DataModule

In [132]:
class RecommenderDataModule(pl.LightningDataModule):
    def __init__(self, interaction_df, graph: KnownledgeGraph, batch_size, val_size):
        super().__init__()
        self.interaction_df = interaction_df
        self.batch_size = batch_size
        self.val_size = val_size
        self.graph = graph

    def prepare_data(self):
        # Load interactions
        df_inter = self.interaction_df

        unique_users = df_inter['user_id'].unique()
        unique_items = df_inter['entity_id'].unique()

        # Build positive interactions
        dataset = []    # user_id, entity_id (item_id), neighbor_ids, relation_ids,
        for _, row in df_inter.iterrows():
            u = row['user_id']
            e = row['entity_id']
            n, r = self.graph.get_neighbors(e)            
            dataset.append((u, e, n, r))


        # Split interactions for validation
        train_size = int(len(dataset) * (1 - self.val_size))
        val_size = len(dataset) - train_size
        self.train_dataset, self.val_dataset = random_split(dataset, [train_size, val_size])

        train_user_pos_items = defaultdict(set)
        for u, i, _, _ in self.train_dataset:
            train_user_pos_items[u].add(i)

        val_user_pos_items = defaultdict(set)
        for u, i, _, _ in self.val_dataset:
            val_user_pos_items[u].add(i)

        self.train_user_pos_items = train_user_pos_items
        self.val_user_pos_items = val_user_pos_items

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)

#### 1.4 Main

In [133]:
tckg_df = pd.read_csv(f'./data/{name}_TCKG.csv')
graph_df = tckg_df[tckg_df['relation_id'] <= 20]

interaction_df = tckg_df[tckg_df['relation_id'] >=21]

interaction_df = interaction_df.rename(columns={'head_id': 'user_id', 'tail_id': 'entity_id'})
unique_items = interaction_df['entity_id'].unique()
pos_item_pool = torch.tensor(unique_items, dtype=torch.long)

graph = KnownledgeGraph(graph_df)
graph.build()

data_module = RecommenderDataModule(interaction_df, graph, batch_size=512, val_size=0.2)
data_module.prepare_data()

In [134]:
from pytorch_lightning import Trainer

model = KGCN(
        graph = graph,
        pos_item_pool = pos_item_pool,
        num_neg_samples = 20,
        embedding_dim= 128,
        lr= 0.02
    )

    # Early stopping callback
early_stop_callback = EarlyStopping(
    monitor="val_loss",     # metric to monitor
    patience=50,            # number of epochs with no improvement after which training will be stopped
    mode="min",             # 'min' because we want to minimize val_loss
    verbose=True
)

checkpoint_hit = ModelCheckpoint(
        monitor="val_hit@10",
        dirpath="./checkpoints",
        filename=f"'movie-NBAgg'-{{epoch:02d}}-{{val_loss:.3f}}{{val_hit@10:.3f}}",
        save_top_k=1,
        mode="max",
)

trainer = Trainer(
        num_sanity_val_steps=0,
        max_epochs=500,
        accelerator="auto",
        callbacks=[checkpoint_hit, early_stop_callback],
        # callbacks=[GradientCheckCallback(), early_stop_callback],
    )

trainer.fit(model, data_module)

Epoch 0: 100%|██████████| 37/37 [07:50<00:00,  0.08it/s, v_num=25, train_loss_step=1.920]


GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name               | Type          | Params | Mode  | FLOPs
---------------------------------------------------------------------
0 | entity_embedding   | Embedding     | 7.0 M  | train | 0    
1 | relation_embedding | Embedding     | 4.6 K  | train | 0    
2 | aggregator         | SumAggregator | 16.5 K | train | 0    
3 | dropout            | Dropout       | 0      | train | 0    
---------------------------------------------------------------------
7.0 M     Trainable params
0         Non-trainable params
7.0 M     Total params
28.007    Total estimated model params size (MB)
5         Modules in train mode
0         Modules in eval mode
0         Total Flops


Epoch 0: 100%|██████████| 37/37 [00:06<00:00,  5.41it/s, v_num=27, train_loss_step=1.920, val_hit@10=0.0638, val_recall@10=0.00709, val_precision@10=0.00638, val_ndcg@10=0.00647, val_loss=1.870, train_loss_epoch=2.620]

Metric val_loss improved. New best score: 1.866


Epoch 1: 100%|██████████| 37/37 [00:07<00:00,  5.25it/s, v_num=27, train_loss_step=0.236, val_hit@10=0.0786, val_recall@10=0.00752, val_precision@10=0.00786, val_ndcg@10=0.00771, val_loss=0.995, train_loss_epoch=0.251]

...

Epoch 16: 100%|██████████| 37/37 [00:06<00:00,  5.39it/s, v_num=27, train_loss_step=0.00158, val_hit@10=0.139, val_recall@10=0.0172, val_precision@10=0.0172, val_ndcg@10=0.0185, val_loss=0.158, train_loss_epoch=0.00134] 

Metric val_loss improved by 0.027 >= min_delta = 0.0. New best score: 0.158


Epoch 17: 100%|██████████| 37/37 [00:03<00:00,  9.26it/s, v_num=27, train_loss_step=0.00126, val_hit@10=0.139, val_recall@10=0.0172, val_precision@10=0.0172, val_ndcg@10=0.0185, val_loss=0.158, train_loss_epoch=0.00134]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1